In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import joblib
import warnings
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import LabelEncoder
from sklearn.base import clone
import optuna
import xgboost as xgb
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor

DATA_PATH = r"A:\College\Thesis\SHD\USstock\comstock\baseline.parquet"
MODEL_SAVE_DIR = Path(r"A:\College\Thesis\Submission\models\comstock_5.5.2")
PLOTS_DIR = MODEL_SAVE_DIR / "plots"
MODEL_SAVE_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
CV_FOLDS = 5
MIN_SAMPLES_PER_ZONE = 250
N_TRIALS_XGB = 30
N_TRIALS_CAT = 30
N_TRIALS_LGB = 30
FEATURE_MODE = "max_accuracy"
USE_LOG_TARGET = True
GEN_GAP_WEIGHT = 0.5
USE_MONOTONIC_CONSTRAINTS = True
TRAIN_GLOBAL = True
TRAIN_PER_ZONE = True


def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


def make_actual_vs_pred_plot(y_true, y_pred, title, out_path):
    plt.figure(figsize=(7, 7))
    plt.scatter(y_true, y_pred, s=8, alpha=0.35)
    mn = min(np.min(y_true), np.min(y_pred))
    mx = max(np.max(y_true), np.max(y_pred))
    plt.plot([mn, mx], [mn, mx], "r--", linewidth=2)
    plt.xlabel("Actual specific heating demand (kWh/m²/yr)")
    plt.ylabel("Predicted specific heating demand (kWh/m²/yr)")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_path, dpi=220)
    plt.close()


def parse_year_from_text(v):
    if pd.isna(v):
        return np.nan
    import re
    s = str(v)
    years = re.findall(r"(19\d{2}|20\d{2})", s)
    if len(years) == 0:
        return np.nan
    return float(max(int(y) for y in years))


def safe_numeric(series):
    return pd.to_numeric(series, errors="coerce")


def add_missing_flags(df, cols):
    for c in cols:
        if c in df.columns:
            df[f"is_missing__{c}"] = df[c].isna().astype(int)
    return df


def fit_label_encoders(X_df):
    X_df = X_df.copy()
    encoders = {}
    for col in X_df.columns:
        s = X_df[col].astype("string").fillna("__MISSING__").astype(str)
        n = pd.to_numeric(s, errors="coerce")
        if n.notna().all():
            X_df[col] = n.astype(np.float64)
        else:
            le = LabelEncoder()
            X_df[col] = le.fit_transform(s).astype(np.float64)
            encoders[col] = le
    return X_df, encoders


def transform_with_encoders(X_df, encoders):
    X_df = X_df.copy()
    for col in X_df.columns:
        s = X_df[col].astype("string").fillna("__MISSING__").astype(str)
        if col in encoders:
            le = encoders[col]
            cls = set(le.classes_)
            vals = []
            for v in s.values:
                if v in cls:
                    vals.append(le.transform([v])[0])
                elif "__MISSING__" in cls:
                    vals.append(le.transform(["__MISSING__"])[0])
                else:
                    vals.append(0)
            X_df[col] = np.array(vals, dtype=np.float64)
        else:
            X_df[col] = pd.to_numeric(s, errors="coerce").fillna(0).astype(np.float64)
    return X_df


def crossval_rmse(model_builder, X, y, n_splits=5, random_state=42, log_target=False):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    scores = []
    for tr_idx, va_idx in kf.split(X):
        X_tr, X_va = X[tr_idx], X[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]
        y_fit = np.log1p(y_tr) if log_target else y_tr
        m = model_builder()
        m.fit(X_tr, y_fit)
        p = m.predict(X_va)
        if log_target:
            p = np.expm1(p)
            p = np.clip(p, 0, None)
        scores.append(rmse(y_va, p))
    return float(np.mean(scores)), float(np.std(scores))


def build_monotone_vectors(features):
    monotone_map = {f: 0 for f in features}
    high_conf_must_increase = {
        "out.params.hdd65f", "out.params.hdd50f", "out.params.hours_below_50f",
        "out.params.hours_below_17f", "out.params.hours_below_0f",
        "in.weekday_operating_hours", "in.weekend_operating_hours", "operating_hours_week",
        "hdd_x_wall_u", "hdd_x_roof_u", "hdd_x_win_u", "hdd_x_wall_area", "hdd_x_window_area"
    }
    for f in high_conf_must_increase:
        if f in monotone_map:
            monotone_map[f] = 1
    xgb_tuple = tuple(monotone_map[f] for f in features)
    lgb_list = [monotone_map[f] for f in features]
    return monotone_map, xgb_tuple, lgb_list


# ── FIX 1: encoders passed as parameter; best_pred tracked inside loop;
#           StackingRegressor receives fresh (unfitted) estimator instances ──
def tune_and_train(X_train, y_train, features, zone_name, encoders, X_test, y_test):
    monotone_map, xgb_mono, lgb_mono = build_monotone_vectors(features)

    def tune_xgb():
        def objective(trial):
            p = {
                "n_estimators": trial.suggest_int("n_estimators", 250, 1200),
                "max_depth": trial.suggest_int("max_depth", 3, 7),
                "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.08, log=True),
                "subsample": trial.suggest_float("subsample", 0.65, 0.95),
                "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 0.9),
                "min_child_weight": trial.suggest_float("min_child_weight", 5.0, 40.0),
                "gamma": trial.suggest_float("gamma", 0.0, 15.0),
                "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 30.0, log=True),
                "reg_lambda": trial.suggest_float("reg_lambda", 1e-2, 60.0, log=True),
                "random_state": RANDOM_STATE,
                "tree_method": "hist",
                "n_jobs": -1,
            }
            if USE_MONOTONIC_CONSTRAINTS:
                p["monotone_constraints"] = xgb_mono

            def builder():
                return xgb.XGBRegressor(**p)

            m, _ = crossval_rmse(builder, X_train, y_train, n_splits=CV_FOLDS,
                                 random_state=RANDOM_STATE, log_target=USE_LOG_TARGET)
            return m

        study = optuna.create_study(direction="minimize")
        study.optimize(objective, n_trials=N_TRIALS_XGB, show_progress_bar=False)
        return study.best_params, study.best_value

    def tune_cat():
        def objective(trial):
            p = {
                "iterations": trial.suggest_int("iterations", 250, 1200),
                "depth": trial.suggest_int("depth", 4, 8),
                "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.08, log=True),
                "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 5.0, 60.0),
                "random_strength": trial.suggest_float("random_strength", 1.0, 12.0),
                "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 6.0),
                "loss_function": "RMSE",
                "random_seed": RANDOM_STATE,
                "verbose": 0,
            }

            def builder():
                return CatBoostRegressor(**p)

            m, _ = crossval_rmse(builder, X_train, y_train, n_splits=CV_FOLDS,
                                 random_state=RANDOM_STATE, log_target=USE_LOG_TARGET)
            return m

        study = optuna.create_study(direction="minimize")
        study.optimize(objective, n_trials=N_TRIALS_CAT, show_progress_bar=False)
        return study.best_params, study.best_value

    def tune_lgb():
        def objective(trial):
            p = {
                "n_estimators": trial.suggest_int("n_estimators", 250, 1200),
                "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.08, log=True),
                "num_leaves": trial.suggest_int("num_leaves", 16, 80),
                "max_depth": trial.suggest_int("max_depth", 3, 8),
                "min_child_samples": trial.suggest_int("min_child_samples", 40, 300),
                "subsample": trial.suggest_float("subsample", 0.65, 0.95),
                "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 0.9),
                "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 30.0, log=True),
                "reg_lambda": trial.suggest_float("reg_lambda", 1e-2, 60.0, log=True),
                "random_state": RANDOM_STATE,
                "n_jobs": -1,
            }
            if USE_MONOTONIC_CONSTRAINTS:
                p["monotone_constraints"] = lgb_mono

            def builder():
                return LGBMRegressor(**p)

            m, _ = crossval_rmse(builder, X_train, y_train, n_splits=CV_FOLDS,
                                 random_state=RANDOM_STATE, log_target=USE_LOG_TARGET)
            return m

        study = optuna.create_study(direction="minimize")
        study.optimize(objective, n_trials=N_TRIALS_LGB, show_progress_bar=False)
        return study.best_params, study.best_value

    xgb_best, xgb_cv = tune_xgb()
    cat_best, cat_cv = tune_cat()
    lgb_best, lgb_cv = tune_lgb()

    print(f"Best CV RMSE -> XGB:{xgb_cv:.3f} | CAT:{cat_cv:.3f} | LGB:{lgb_cv:.3f}")

    xgb_final_params = dict(xgb_best)
    lgb_final_params = dict(lgb_best)
    if USE_MONOTONIC_CONSTRAINTS:
        xgb_final_params["monotone_constraints"] = xgb_mono
        lgb_final_params["monotone_constraints"] = lgb_mono

    # Fitted models used for individual evaluation and saving
    xgb_model = xgb.XGBRegressor(**xgb_final_params, random_state=RANDOM_STATE,
                                  tree_method="hist", n_jobs=-1)
    cat_model = CatBoostRegressor(**cat_best, random_seed=RANDOM_STATE, verbose=0)
    lgb_model = LGBMRegressor(**lgb_final_params, random_state=RANDOM_STATE, n_jobs=-1)

    y_train_fit = np.log1p(y_train) if USE_LOG_TARGET else y_train

    xgb_model.fit(X_train, y_train_fit)
    cat_model.fit(X_train, y_train_fit, verbose=False)
    lgb_model.fit(X_train, y_train_fit)

    # FIX 1: Fresh (unfitted) instances for StackingRegressor to avoid
    # sklearn _estimator_type validation error on newer sklearn versions.
    xgb_stack = xgb.XGBRegressor(**xgb_final_params, random_state=RANDOM_STATE,
                                  tree_method="hist", n_jobs=-1)
    cat_stack = CatBoostRegressor(**cat_best, random_seed=RANDOM_STATE, verbose=0)
    lgb_stack = LGBMRegressor(**lgb_final_params, random_state=RANDOM_STATE, n_jobs=-1)

    stack_model = StackingRegressor(
        estimators=[("xgb", xgb_stack), ("cat", cat_stack), ("lgb", lgb_stack)],
        final_estimator=Ridge(alpha=2.0),
        cv=5,
        n_jobs=-1,
        passthrough=False,
    )
    stack_model.fit(X_train, y_train_fit)

    models = [
        ("XGBoost", xgb_model),
        ("CatBoost", cat_model),
        ("LightGBM", lgb_model),
        ("Stacking", stack_model),
    ]

    def predict_back(m, X):
        p = m.predict(X)
        if USE_LOG_TARGET:
            p = np.expm1(p)
            p = np.clip(p, 0, None)
        return p

    safe_zone = str(zone_name).replace(" ", "_").replace("-", "_").replace("/", "_")

    stats = {}
    rows = []
    best_pred = None  # FIX 2: initialise so it is always defined

    for n, m in models:
        p_tr = predict_back(m, X_train)
        p_te = predict_back(m, X_test)
        rmse_tr = rmse(y_train, p_tr)
        rmse_te = rmse(y_test, p_te)
        gap = rmse_te - rmse_tr
        mae = mean_absolute_error(y_test, p_te)
        r2 = r2_score(y_test, p_te)
        gen_score = rmse_te + GEN_GAP_WEIGHT * max(gap, 0.0)
        print(f"{n:9s} | MAE:{mae:8.3f} | RMSE:{rmse_te:8.3f} | R²:{r2:7.4f} | "
              f"Gap:{gap:7.3f} | GenScore:{gen_score:8.3f}")
        stats[n] = {"rmse": rmse_te, "gap": gap, "gen_score": gen_score}

        # Save plot for every model
        plot_path = PLOTS_DIR / f"actual_vs_pred_{safe_zone}_{n.replace(' ', '_')}.png"
        make_actual_vs_pred_plot(y_test, p_te, f"{zone_name} — {n}", plot_path)

        rows.append({
            "zone":       zone_name,
            "model":      n,
            "n_train":    len(y_train),
            "n_test":     len(y_test),
            "mae":        round(mae, 4),
            "rmse_train": round(rmse_tr, 4),
            "rmse_test":  round(rmse_te, 4),
            "r2":         round(r2, 4),
            "gap":        round(gap, 4),
            "gen_score":  round(gen_score, 4),
        })

    # Write / append CSV
    csv_path = MODEL_SAVE_DIR / "evaluation_results.csv"
    results_df = pd.DataFrame(rows)
    if csv_path.exists():
        results_df.to_csv(csv_path, mode="a", header=False, index=False)
    else:
        results_df.to_csv(csv_path, index=False)
    print(f"Metrics saved → {csv_path}")

    best_name = min(stats.items(), key=lambda kv: kv[1]["gen_score"])[0]

    baseline_pred = np.full_like(y_test, np.median(y_train), dtype=np.float64)
    baseline_rmse = rmse(y_test, baseline_pred)
    print(f"Baseline RMSE: {baseline_rmse:.3f}")
    print(f"Best model: {best_name}")

    # FIX 3: removed duplicate make_actual_vs_pred_plot call that used undefined best_pred.
    # All per-model plots are already saved inside the loop above.

    joblib.dump(xgb_model,   MODEL_SAVE_DIR / f"xgb_{safe_zone}.joblib")
    joblib.dump(cat_model,   MODEL_SAVE_DIR / f"catboost_{safe_zone}.joblib")
    joblib.dump(lgb_model,   MODEL_SAVE_DIR / f"lightgbm_{safe_zone}.joblib")
    joblib.dump(stack_model, MODEL_SAVE_DIR / f"stacking_{safe_zone}.joblib")
    joblib.dump(encoders,    MODEL_SAVE_DIR / f"encoders_{safe_zone}.joblib")  # FIX 3: now in scope
    joblib.dump(features,    MODEL_SAVE_DIR / f"features_{safe_zone}.joblib")

    return best_name


# ── Data loading ─────────────────────────────────────────────────────────────
print("=== Loading ComStock parquet ===")
df = pd.read_parquet(DATA_PATH).convert_dtypes()
print(f"Raw shape: {df.shape}")

alias_hits = {}


def map_first_match(df, canonical, exact_names=None, contains_any=None):
    if canonical in df.columns:
        return canonical
    exact_names = exact_names or []
    contains_any = contains_any or []
    for c in exact_names:
        if c in df.columns:
            df[canonical] = df[c]
            return c
    for c in df.columns:
        cl = c.lower()
        if all(tok in cl for tok in contains_any):
            df[canonical] = df[c]
            return c
    return None


alias_hits["in.weekday_operating_hours"] = map_first_match(
    df, "in.weekday_operating_hours",
    exact_names=["in.weekday_operating_hours", "in.weekday_operating_hours..hr"],
    contains_any=["weekday", "operating", "hour"])
alias_hits["in.weekend_operating_hours"] = map_first_match(
    df, "in.weekend_operating_hours",
    exact_names=["in.weekend_operating_hours", "in.weekend_operating_hours..hr"],
    contains_any=["weekend", "operating", "hour"])
alias_hits["in.weekday_opening_time"] = map_first_match(
    df, "in.weekday_opening_time",
    exact_names=["in.weekday_opening_time", "in.weekday_opening_time..hr"],
    contains_any=["weekday", "open", "time"])
alias_hits["in.weekend_opening_time"] = map_first_match(
    df, "in.weekend_opening_time",
    exact_names=["in.weekend_opening_time", "in.weekend_opening_time..hr"],
    contains_any=["weeke", "open", "time"])
alias_hits["out.params.average_heating_setpoint_max"] = map_first_match(
    df, "out.params.average_heating_setpoint_max",
    exact_names=["out.params.average_heating_setpoint_max",
                 "out.params.average_heating_setpoint_max..c"],
    contains_any=["average", "heating", "setpoint", "max"])
alias_hits["out.params.average_heating_setpoint_min"] = map_first_match(
    df, "out.params.average_heating_setpoint_min",
    exact_names=["out.params.average_heating_setpoint_min",
                 "out.params.average_heating_setpoint_min..c"],
    contains_any=["average", "heating", "setpoint", "min"])
alias_hits["out.params.hours_heating_setpoint_not_met"] = map_first_match(
    df, "out.params.hours_heating_setpoint_not_met",
    exact_names=["out.params.hours_heating_setpoint_not_met",
                 "out.params.hours_heating_setpoint_not_met..hr"],
    contains_any=["hours", "heating", "setpoint", "not", "met"])

if "calc.enduse_group.site_energy.heating.energy_consumption" in df.columns:
    df["heating_energy_kwh"] = safe_numeric(
        df["calc.enduse_group.site_energy.heating.energy_consumption"])
else:
    parts = ["out.electricity.heating.energy_consumption",
             "out.natural_gas.heating.energy_consumption",
             "out.other_fuel.heating.energy_consumption",
             "out.district_heating.heating.energy_consumption"]
    s = 0
    for c in parts:
        s = s + safe_numeric(df[c]).fillna(0.0)
    df["heating_energy_kwh"] = s

if "in.sqft" in df.columns:
    df["floor_area_ft2"] = safe_numeric(df["in.sqft"])
elif "calc.weighted.sqft" in df.columns:
    df["floor_area_ft2"] = safe_numeric(df["calc.weighted.sqft"])

df = df.dropna(subset=["heating_energy_kwh", "floor_area_ft2"]).copy()
df = df[(df["floor_area_ft2"] > 0) & (df["heating_energy_kwh"] >= 0)].copy()
df["floor_area_m2"] = df["floor_area_ft2"] * 0.09290304
df["specific_heat_demand_kwh_m2"] = df["heating_energy_kwh"] / df["floor_area_m2"]
df = df[np.isfinite(df["specific_heat_demand_kwh_m2"])].copy()
df = df[df["specific_heat_demand_kwh_m2"] > 0].copy()
q1 = df["specific_heat_demand_kwh_m2"].quantile(0.005)
q2 = df["specific_heat_demand_kwh_m2"].quantile(0.995)
df = df[(df["specific_heat_demand_kwh_m2"] >= q1) & (df["specific_heat_demand_kwh_m2"] <= q2)].copy()

if "upgrade" in df.columns:
    u = df["upgrade"].astype("string").str.strip().str.lower()
    is_baseline = (u.isin(["0", "00", "baseline", "base", "upgrade00"]) |
                   u.str.fullmatch(r"0+"))
    if is_baseline.sum() > 1000:
        df = df[is_baseline].copy()

CLIMATE_COL = "in.building_america_climate_zone"

deployable_features = [
    "in.comstock_building_type", "in.comstock_building_type_group",
    "in.building_subtype", "in.floor_area_category", "in.number_of_stories",
    "in.aspect_ratio", "in.rotation", "in.wall_construction_type",
    "in.window_to_wall_ratio_category", "in.window_type", "in.vintage",
    "in.year_built", CLIMATE_COL, "in.ashrae_iecc_climate_zone_2006",
    "in.state", "in.census_region_name", "in.census_division_name",
    "in.cambium_grid_region", "in.iso_rto_region", "in.cluster_id",
    "in.cluster_name", "in.county_name", "in.weather_file_2018",
    "in.weather_file_tmy3", "in.hvac_system_type", "in.hvac_category",
    "in.hvac_heat_type", "in.hvac_cool_type", "in.hvac_vent_type",
    "in.hvac_combined_type", "in.heating_fuel", "in.hvac_night_variability",
    "in.weekday_opening_time", "in.weekday_operating_hours",
    "in.weekend_opening_time", "in.weekend_operating_hours",
    "in.energy_code_followed_during_original_building_construction",
    "in.energy_code_followed_during_last_hvac_replacement",
    "in.energy_code_followed_during_last_roof_replacement",
    "in.energy_code_followed_during_last_walls_replacement",
    "in.ownership_type", "in.party_responsible_for_operation",
    "in.purchase_input_responsibility",
]
max_accuracy_additional = [
    "out.params.hdd65f", "out.params.hdd50f", "out.params.cdd65f",
    "out.params.hours_below_50f", "out.params.hours_below_17f",
    "out.params.hours_below_0f",
    "out.params.average_wall_u_value..btu_per_ft2_f_hr",
    "out.params.average_roof_u_value", "out.params.average_window_u_value",
    "out.params.average_window_shgc", "out.params.window_to_wall_ratio",
    "out.params.ext_wall_area", "out.params.ext_roof_area",
    "out.params.ext_window_area", "out.params.building_fraction_heated",
    "out.params.heating_equipment", "out.params.boiler_average_efficiency",
    "out.params.boiler_capacity", "out.params.dx_heating_average_cop",
    "out.params.dx_heating_design_cop", "out.params.heat_pump_heating_average_cop",
    "out.params.heat_pump_heating_capacity",
    "out.params.average_heating_setpoint_max",
    "out.params.average_heating_setpoint_min",
    "out.params.hours_heating_setpoint_not_met",
    "out.params.average_outdoor_air_fraction",
    "out.params.occupant_density_ppl_per_m_2",
    "out.params.occupant_eflh", "out.params.interior_equipment_power_density",
    "out.params.interior_electric_equipment_eflh",
    "out.params.interior_lighting_power_density",
    "out.params.interior_lighting_eflh",
]

candidate_features = list(deployable_features)
if FEATURE_MODE == "max_accuracy":
    candidate_features += max_accuracy_additional
candidate_features = [c for c in candidate_features if c in df.columns]

for c in ["in.number_of_stories", "in.aspect_ratio", "in.rotation",
          "in.weekday_opening_time", "in.weekday_operating_hours..hr",
          "in.weekend_opening_time", "in.weekend_operating_hours",
          "out.params.average_heating_setpoint_max",
          "out.params.average_heating_setpoint_min"]:
    if c in df.columns:
        df[c] = safe_numeric(df[c])

if "in.year_built" in df.columns:
    df["year_built_num"] = df["in.year_built"].apply(parse_year_from_text)

if {"in.weekday_operating_hours..hr", "in.weekend_operating_hours"}.issubset(df.columns):
    wkd = safe_numeric(df["in.weekday_operating_hours..hr"])
    wke = safe_numeric(df["in.weekend_operating_hours..hr"])
    df["operating_hours_week"] = 5.0 * wkd + 2.0 * wke
    df["is_missing__in.weekday_operating_hours..hr"] = wkd.isna().astype(int)
    df["is_missing__in.weekend_operating_hours"] = wke.isna().astype(int)

if {"out.params.average_heating_setpoint_max..c",
        "out.params.average_heating_setpoint_min..c"}.issubset(df.columns):
    tmax = safe_numeric(df["out.params.average_heating_setpoint_max..c"])
    tmin = safe_numeric(df["out.params.average_heating_setpoint_min..c"])
    df["heating_setpoint_span_c"] = tmax - tmin
    df["is_missing__out.params.average_heating_setpoint_max..c"] = tmax.isna().astype(int)
    df["is_missing__out.params.average_heating_setpoint_min..c"] = tmin.isna().astype(int)

if "out.params.hdd65f" in df.columns:
    hdd = safe_numeric(df["out.params.hdd65f"])
    if "out.params.average_wall_u_value..btu_per_ft2_f_hr" in df.columns:
        df["hdd_x_wall_u"] = hdd * safe_numeric(df["out.params.average_wall_u_value..btu_per_ft2_f_hr"])
    if "out.params.average_roof_u_value" in df.columns:
        df["hdd_x_roof_u"] = hdd * safe_numeric(df["out.params.average_roof_u_value"])
    if "out.params.average_window_u_value" in df.columns:
        df["hdd_x_win_u"] = hdd * safe_numeric(df["out.params.average_window_u_value"])
    if "out.params.ext_wall_area" in df.columns:
        df["hdd_x_wall_area"] = hdd * safe_numeric(df["out.params.ext_wall_area"])
    if "out.params.ext_window_area" in df.columns:
        df["hdd_x_window_area"] = hdd * safe_numeric(df["out.params.ext_window_area"])

engineered = [
    "floor_area_ft2", "floor_area_m2", "year_built_num", "operating_hours_week",
    "heating_setpoint_span_c", "hdd_x_wall_u", "hdd_x_roof_u", "hdd_x_win_u",
    "hdd_x_wall_area", "hdd_x_window_area",
]
missing_flag_candidates = [
    "out.params.average_wall_u_value..btu_per_ft2_f_hr",
    "out.params.average_roof_u_value",
    "out.params.average_window_u_value",
    "out.params.hdd65f",
    "in.hvac_system_type",
    "in.window_type",
    "in.weekday_operating_hours..hr",
    "in.weekend_operating_hours..hr",
    "out.params.average_heating_setpoint_max..c",
    "out.params.average_heating_setpoint_min..c",
    "out.params.hours_heating_setpoint_not_met",
]
df = add_missing_flags(df, missing_flag_candidates)
engineered += [c for c in df.columns if c.startswith("is_missing__")]

features = candidate_features + [c for c in engineered if c in df.columns]
features = list(dict.fromkeys(features))

df = df.dropna(subset=["specific_heat_demand_kwh_m2", CLIMATE_COL]).copy()
df[features] = df[features].astype("string").fillna("__MISSING__")

print(f"Rows ready: {len(df)}")
print(f"Total features: {len(features)}")

# ── Global train/test split ───────────────────────────────────────────────────
X_raw = df[features].copy()
y = df["specific_heat_demand_kwh_m2"].to_numpy(dtype=np.float64)

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.20, random_state=RANDOM_STATE)
X_train_df, encoders = fit_label_encoders(X_train_raw)
X_test_df = transform_with_encoders(X_test_raw, encoders)
X_train = X_train_df.to_numpy(dtype=np.float64)
X_test = X_test_df.to_numpy(dtype=np.float64)

# ── Train global model ────────────────────────────────────────────────────────
if TRAIN_GLOBAL:
    print("\n=== Training Global Model ===")
    # FIX: pass encoders, X_test, y_test explicitly — no more global mutation
    tune_and_train(X_train, y_train, features, "global", encoders, X_test, y_test)

# ── Train per-zone models ─────────────────────────────────────────────────────
if TRAIN_PER_ZONE:
    zones = sorted(df[CLIMATE_COL].astype("string").dropna().unique().tolist())
    for zone in zones:
        d = df[df[CLIMATE_COL].astype("string") == zone].copy()
        if len(d) < MIN_SAMPLES_PER_ZONE:
            continue
        print(f"\n=== Zone: {zone} | samples={len(d)} ===")
        X_raw_z = d[features].copy()
        y_z = d["specific_heat_demand_kwh_m2"].to_numpy(dtype=np.float64)
        X_tr_raw, X_te_raw, y_tr, y_te = train_test_split(
            X_raw_z, y_z, test_size=0.20, random_state=RANDOM_STATE)
        X_tr_df, enc_z = fit_label_encoders(X_tr_raw)
        X_te_df = transform_with_encoders(X_te_raw, enc_z)
        X_tr = X_tr_df.to_numpy(dtype=np.float64)
        X_te = X_te_df.to_numpy(dtype=np.float64)
        # FIX: pass enc_z, X_te, y_te explicitly — no global mutation needed
        tune_and_train(X_tr, y_tr, features, zone, enc_z, X_te, y_te)

print("\n=== TRAINING COMPLETE ===")
print(f"Output folder: {MODEL_SAVE_DIR}")

In [30]:
import numpy as np
import pandas as pd
from pathlib import Path
import joblib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings("ignore")

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

DATA_PATH = r"A:\College\Thesis\SHD\USstock\comstock\baseline.parquet"
MODEL_SAVE_DIR = Path(r"A:\College\Thesis\Submission\models\comstock_5.5")
EVAL_DIR = MODEL_SAVE_DIR / "evaluation"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

CLIMATE_COL = "in.building_america_climate_zone"
RANDOM_STATE = 42
USE_LOG_TARGET = True   # ← must match training
APPLY_TARGET_TRIM = True
TRIM_Q_LOW = 0.005
TRIM_Q_HIGH = 0.995

# ─────────────────────────────────────────────
# Helpers
# ─────────────────────────────────────────────
def safe_numeric(series):
    return pd.to_numeric(series, errors="coerce")

def parse_year_from_text(v):
    if pd.isna(v):
        return np.nan
    import re
    s = str(v)
    years = re.findall(r"(19\d{2}|20\d{2})", s)
    return np.nan if not years else float(max(int(y) for y in years))

def add_missing_flags(df, cols):
    for c in cols:
        if c in df.columns:
            df[f"is_missing__{c}"] = df[c].isna().astype(int)
    return df

def transform_with_encoders(X_df, encoders):
    X_df = X_df.copy()
    for col in X_df.columns:
        s = X_df[col].astype("string").fillna("__MISSING__").astype(str)
        if col in encoders:
            le = encoders[col]
            cls = set(le.classes_)
            vals = []
            for v in s.values:
                if v in cls:
                    vals.append(le.transform([v])[0])
                elif "__MISSING__" in cls:
                    vals.append(le.transform(["__MISSING__"])[0])
                else:
                    vals.append(0)
            X_df[col] = np.array(vals, dtype=np.float64)
        else:
            X_df[col] = pd.to_numeric(s, errors="coerce").fillna(0).astype(np.float64)
    return X_df

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def predict_back(model, X):
    """Predict and invert log1p transform if used during training."""
    p = model.predict(X)
    if USE_LOG_TARGET:
        p = np.expm1(p)
        p = np.clip(p, 0, None)
    return p

# ─────────────────────────────────────────────
# XGBoost version-mismatch shim
#
# Models were pickled with an older XGBoost that
# stored params like gpu_id, use_label_encoder,
# etc. in __dict__. XGBoost >=2.x removed them,
# so get_params() (called internally by sklearn's
# StackingRegressor.predict → transform) raises
# AttributeError.
#
# Fix: monkey-patch XGBModel.__getattr__ to return
# None for any removed legacy param name so that
# get_params() succeeds. We use a whitelist of
# known-removed names so we never silence real
# functional attributes like best_iteration.
# ─────────────────────────────────────────────
_XGB_REMOVED_PARAMS = frozenset({
    "gpu_id", "gpu_hist", "use_label_encoder",
    "interaction_constraints", "monotone_constraints",
    "num_parallel_tree", "validate_parameters",
    "enable_categorical", "feature_types",
    "max_cat_to_onehot", "max_cat_threshold",
    "grow_policy", "max_leaves", "max_bin",
    "sampling_method", "sketch_eps", "updater",
    "refresh_leaf", "process_type", "predictor",
    "single_precision_histogram", "deterministic_histogram",
    "callbacks", "early_stopping_rounds", "eval_metric",
    "verbosity", "importance_type",
})

def _apply_xgb_compat_shim():
    try:
        from xgboost.sklearn import XGBModel
    except ImportError:
        return  # XGBoost not installed, nothing to do

    if getattr(XGBModel, "_compat_shim_applied", False):
        return  # already patched, don't double-patch

    _original_getattr = XGBModel.__dict__.get("__getattr__", None)

    def _shim_getattr(self, name):
        if name in _XGB_REMOVED_PARAMS:
            return None   # satisfy get_params() without crashing
        # For all other missing attributes, raise normally so that
        # real errors (e.g. accessing best_iteration before fit) surface.
        raise AttributeError(
            f"'{type(self).__name__}' object has no attribute '{name}'"
        )

    XGBModel.__getattr__ = _shim_getattr
    XGBModel._compat_shim_applied = True

_apply_xgb_compat_shim()

# ─────────────────────────────────────────────
# Model loading
# ─────────────────────────────────────────────
def load_model_bundle(zone_safe):
    feats_path = MODEL_SAVE_DIR / f"features_{zone_safe}.joblib"
    enc_path   = MODEL_SAVE_DIR / f"encoders_{zone_safe}.joblib"
    if not feats_path.exists() or not enc_path.exists():
        raise FileNotFoundError(f"Missing features/encoders for zone '{zone_safe}'")
    features = joblib.load(feats_path)
    encoders = joblib.load(enc_path)
    models = {}
    for key, fname in [
        ("XGBoost",  f"xgb_{zone_safe}.joblib"),
        ("CatBoost", f"catboost_{zone_safe}.joblib"),
        ("LightGBM", f"lightgbm_{zone_safe}.joblib"),
        ("Stacking", f"stacking_{zone_safe}.joblib"),
    ]:
        p = MODEL_SAVE_DIR / fname
        if not p.exists():
            continue
        models[key] = joblib.load(p)
    if not models:
        raise FileNotFoundError(f"No model files found for zone '{zone_safe}'")
    return features, encoders, models

def zone_to_safe(zone_name):
    return str(zone_name).replace(" ", "_").replace("-", "_").replace("/", "_")

# ─────────────────────────────────────────────
# Feature engineering  (mirrors training exactly)
# ─────────────────────────────────────────────
def build_targets_and_minimal_features(df):
    def map_first_match(df, canonical, exact_names=None, contains_any=None):
        if canonical in df.columns:
            return canonical
        for c in (exact_names or []):
            if c in df.columns:
                df[canonical] = df[c]
                return c
        for c in df.columns:
            if all(tok in c.lower() for tok in (contains_any or [])):
                df[canonical] = df[c]
                return c
        return None

    map_first_match(df, "in.weekday_operating_hours",
                    exact_names=["in.weekday_operating_hours", "in.weekday_operating_hours..hr"],
                    contains_any=["weekday", "operating", "hour"])
    map_first_match(df, "in.weekend_operating_hours",
                    exact_names=["in.weekend_operating_hours", "in.weekend_operating_hours..hr"],
                    contains_any=["weekend", "operating", "hour"])
    map_first_match(df, "out.params.average_heating_setpoint_max",
                    exact_names=["out.params.average_heating_setpoint_max",
                                 "out.params.average_heating_setpoint_max..c"],
                    contains_any=["average", "heating", "setpoint", "max"])
    map_first_match(df, "out.params.average_heating_setpoint_min",
                    exact_names=["out.params.average_heating_setpoint_min",
                                 "out.params.average_heating_setpoint_min..c"],
                    contains_any=["average", "heating", "setpoint", "min"])

    if "calc.enduse_group.site_energy.heating.energy_consumption" in df.columns:
        df["heating_energy_kwh"] = safe_numeric(
            df["calc.enduse_group.site_energy.heating.energy_consumption"])
    else:
        parts = ["out.electricity.heating.energy_consumption",
                 "out.natural_gas.heating.energy_consumption",
                 "out.other_fuel.heating.energy_consumption",
                 "out.district_heating.heating.energy_consumption"]
        s = 0
        for c in parts:
            if c in df.columns:
                s = s + safe_numeric(df[c]).fillna(0.0)
        df["heating_energy_kwh"] = s

    if "in.sqft" in df.columns:
        df["floor_area_ft2"] = safe_numeric(df["in.sqft"])
    elif "calc.weighted.sqft" in df.columns:
        df["floor_area_ft2"] = safe_numeric(df["calc.weighted.sqft"])

    df = df.dropna(subset=["heating_energy_kwh", "floor_area_ft2"]).copy()
    df = df[(df["floor_area_ft2"] > 0) & (df["heating_energy_kwh"] >= 0)].copy()
    df["floor_area_m2"]               = df["floor_area_ft2"] * 0.09290304
    df["specific_heat_demand_kwh_m2"] = df["heating_energy_kwh"] / df["floor_area_m2"]
    df = df[np.isfinite(df["specific_heat_demand_kwh_m2"])].copy()
    df = df[df["specific_heat_demand_kwh_m2"] > 0].copy()

    if APPLY_TARGET_TRIM:
        q1 = df["specific_heat_demand_kwh_m2"].quantile(TRIM_Q_LOW)
        q2 = df["specific_heat_demand_kwh_m2"].quantile(TRIM_Q_HIGH)
        df = df[(df["specific_heat_demand_kwh_m2"] >= q1) &
                (df["specific_heat_demand_kwh_m2"] <= q2)].copy()

    if "in.year_built" in df.columns:
        df["year_built_num"] = df["in.year_built"].apply(parse_year_from_text)

    for c in ["in.number_of_stories", "in.aspect_ratio", "in.rotation",
              "in.weekday_operating_hours", "in.weekend_operating_hours",
              "out.params.average_heating_setpoint_max", "out.params.average_heating_setpoint_min"]:
        if c in df.columns:
            df[c] = safe_numeric(df[c])

    if {"in.weekday_operating_hours", "in.weekend_operating_hours"}.issubset(df.columns):
        wkd = safe_numeric(df["in.weekday_operating_hours"])
        wke = safe_numeric(df["in.weekend_operating_hours"])
        df["operating_hours_week"] = 5.0 * wkd + 2.0 * wke
        df["is_missing__in.weekday_operating_hours"] = wkd.isna().astype(int)
        df["is_missing__in.weekend_operating_hours"] = wke.isna().astype(int)

    if {"out.params.average_heating_setpoint_max",
        "out.params.average_heating_setpoint_min"}.issubset(df.columns):
        tmax = safe_numeric(df["out.params.average_heating_setpoint_max"])
        tmin = safe_numeric(df["out.params.average_heating_setpoint_min"])
        df["heating_setpoint_span_c"] = tmax - tmin
        df["is_missing__out.params.average_heating_setpoint_max"] = tmax.isna().astype(int)
        df["is_missing__out.params.average_heating_setpoint_min"] = tmin.isna().astype(int)

    if "out.params.hdd65f" in df.columns:
        hdd = safe_numeric(df["out.params.hdd65f"])
        for col, key in [
            ("out.params.average_wall_u_value..btu_per_ft2_f_hr", "hdd_x_wall_u"),
            ("out.params.average_roof_u_value",  "hdd_x_roof_u"),
            ("out.params.average_window_u_value", "hdd_x_win_u"),
            ("out.params.ext_wall_area",   "hdd_x_wall_area"),
            ("out.params.ext_window_area", "hdd_x_window_area"),
        ]:
            if col in df.columns:
                df[key] = hdd * safe_numeric(df[col])

    df = add_missing_flags(df, [
        "out.params.average_wall_u_value..btu_per_ft2_f_hr", "out.params.average_roof_u_value",
        "out.params.average_window_u_value", "out.params.hdd65f", "in.hvac_system_type",
        "in.window_type", "in.weekday_operating_hours..hr", "in.weekend_operating_hours..hr",
        "out.params.average_heating_setpoint_max..c", "out.params.average_heating_setpoint_min..c",
        "out.params.hours_heating_setpoint_not_met"
    ])
    return df

# ─────────────────────────────────────────────
# Plots
# ─────────────────────────────────────────────
def save_eval_plots(y, y_pred, name, zone_safe, label):
    fig = plt.figure(figsize=(18, 5))
    gs  = gridspec.GridSpec(1, 3, figure=fig)

    # 1. Actual vs Predicted
    ax1 = fig.add_subplot(gs[0])
    ax1.scatter(y, y_pred, s=6, alpha=0.3, rasterized=True)
    mn, mx = min(y.min(), y_pred.min()), max(y.max(), y_pred.max())
    ax1.plot([mn, mx], [mn, mx], "r--", lw=1.5)
    ax1.set_xlabel("Actual (kWh/m²/yr)")
    ax1.set_ylabel("Predicted (kWh/m²/yr)")
    ax1.set_title("Actual vs Predicted")

    # 2. Residuals vs Predicted
    ax2 = fig.add_subplot(gs[1])
    resid = y_pred - y
    ax2.scatter(y_pred, resid, s=6, alpha=0.3, rasterized=True)
    ax2.axhline(0, color="r", lw=1.5, ls="--")
    ax2.set_xlabel("Predicted (kWh/m²/yr)")
    ax2.set_ylabel("Residual (Pred − Actual)")
    ax2.set_title("Residuals")

    # 3. Error distribution
    ax3 = fig.add_subplot(gs[2])
    ax3.hist(resid, bins=80, alpha=0.85, color="steelblue", edgecolor="none")
    ax3.axvline(0, color="r", lw=1.5, ls="--")
    ax3.axvline(resid.mean(), color="orange", lw=1.2, ls="--", label=f"mean={resid.mean():.1f}")
    ax3.set_xlabel("Error (kWh/m²/yr)")
    ax3.set_ylabel("Count")
    ax3.set_title("Error Distribution")
    ax3.legend(fontsize=8)

    fig.suptitle(f"{label} — {name}", fontsize=13, fontweight="bold")
    fig.tight_layout()

    out = EVAL_DIR / f"{zone_safe}__{name.replace(' ', '_')}.png"
    fig.savefig(out, dpi=180, bbox_inches="tight")
    plt.close(fig)
    return out

# ─────────────────────────────────────────────
# Core evaluation
# ─────────────────────────────────────────────
def evaluate_bundle(df_eval, features, encoders, models, zone_safe, label):
    missing = [f for f in features if f not in df_eval.columns]
    if missing:
        print(f"  [diag] {len(missing)} features zero-filled: {missing[:8]}")

    X_raw = df_eval.reindex(columns=features).copy()
    X_raw = X_raw.astype("string").fillna("__MISSING__")
    X_df  = transform_with_encoders(X_raw, encoders)
    X     = X_df.to_numpy(dtype=np.float64)
    y     = df_eval["specific_heat_demand_kwh_m2"].to_numpy(dtype=np.float64)

    baseline_rmse_val = rmse(y, np.full_like(y, np.median(y)))

    print(f"\n{'─'*60}")
    print(f"  {label}")
    print(f"  Samples: {len(y):,} | y [{y.min():.1f}, {y.max():.1f}] "
          f"| Baseline RMSE: {baseline_rmse_val:.3f}")
    print(f"{'─'*60}")

    rows = []
    for name, model in models.items():
        y_pred = predict_back(model, X)

        mae_v  = float(mean_absolute_error(y, y_pred))
        rmse_v = rmse(y, y_pred)
        r2_v   = float(r2_score(y, y_pred))
        mape_v = float(np.mean(np.abs((y - y_pred) / np.clip(y, 1e-6, None))) * 100)
        bias_v = float(np.mean(y_pred - y))
        p90_v  = float(np.percentile(np.abs(y_pred - y), 90))

        print(f"  {name:9s} | MAE: {mae_v:7.3f} | RMSE: {rmse_v:7.3f} | "
              f"R²: {r2_v:6.4f} | MAPE: {mape_v:5.1f}% | "
              f"Bias: {bias_v:+7.2f} | P90 err: {p90_v:7.2f}")

        plot_path = save_eval_plots(y, y_pred, name, zone_safe, label)

        rows.append({
            "zone":          label,
            "zone_safe":     zone_safe,
            "model":         name,
            "n_samples":     len(y),
            "mae":           round(mae_v,  4),
            "rmse":          round(rmse_v, 4),
            "r2":            round(r2_v,   4),
            "mape_pct":      round(mape_v, 3),
            "bias":          round(bias_v, 4),
            "p90_abs_error": round(p90_v,  4),
            "baseline_rmse": round(baseline_rmse_val, 4),
            "rmse_vs_baseline": round(rmse_v - baseline_rmse_val, 4),
            "plot":          str(plot_path),
        })

    return rows

# ─────────────────────────────────────────────
# Main
# ─────────────────────────────────────────────
print("=== Loading parquet ===")
df = pd.read_parquet(DATA_PATH).convert_dtypes()
print(f"Raw shape: {df.shape}")

df = build_targets_and_minimal_features(df)

if "upgrade" in df.columns:
    u = df["upgrade"].astype("string").str.strip().str.lower()
    is_baseline = (u.isin(["0", "00", "baseline", "base", "upgrade00"]) |
                   u.str.fullmatch(r"0+"))
    if is_baseline.sum() > 1000:
        df = df[is_baseline].copy()

df = df.dropna(subset=["specific_heat_demand_kwh_m2", CLIMATE_COL]).copy()
print(f"Rows ready for evaluation: {len(df):,}")

# Discover bundles
bundles = ["global"]
for p in MODEL_SAVE_DIR.glob("features_*.joblib"):
    z = p.stem.replace("features_", "", 1)
    if z != "global" and z not in bundles:
        bundles.append(z)
print(f"Found model bundles: {bundles}")

zones_in_data = sorted(df[CLIMATE_COL].astype("string").dropna().unique().tolist())
zone_safe_map = {zone_to_safe(z): z for z in zones_in_data}

all_rows = []

# Global model on all data
features_g, enc_g, models_g = load_model_bundle("global")
rows = evaluate_bundle(df, features_g, enc_g, models_g,
                       zone_safe="global", label="GLOBAL — all rows")
all_rows.extend(rows)

# Per-zone models on their respective rows
for zone_safe in bundles:
    if zone_safe == "global":
        continue
    if zone_safe not in zone_safe_map:
        print(f"\nSkipping '{zone_safe}' (no matching rows in data).")
        continue
    zone_name = zone_safe_map[zone_safe]
    df_z = df[df[CLIMATE_COL].astype("string") == zone_name].copy()
    if len(df_z) == 0:
        continue
    features_z, enc_z, models_z = load_model_bundle(zone_safe)
    rows = evaluate_bundle(df_z, features_z, enc_z, models_z,
                           zone_safe=zone_safe, label=f"ZONE: {zone_name}")
    all_rows.extend(rows)

# ── Save CSV ──────────────────────────────────
results_df = pd.DataFrame(all_rows)
csv_path = EVAL_DIR / "evaluation_results.csv"
results_df.to_csv(csv_path, index=False)
print(f"\n✓ CSV saved → {csv_path}")

# ── Summary table ─────────────────────────────
print("\n=== SUMMARY (Stacking model per zone) ===")
stacking = results_df[results_df["model"] == "Stacking"][
    ["zone", "n_samples", "mae", "rmse", "r2", "mape_pct", "bias", "baseline_rmse"]
].copy()
print(stacking.to_string(index=False))

# ── Cross-zone comparison chart ───────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
model_colors = {"XGBoost": "#4C72B0", "CatBoost": "#DD8452",
                "LightGBM": "#55A868", "Stacking": "#C44E52"}

for ax, metric, ylabel in zip(
    axes,
    ["r2", "rmse", "mae"],
    ["R²", "RMSE (kWh/m²/yr)", "MAE (kWh/m²/yr)"]
):
    for model_name, grp in results_df.groupby("model"):
        ax.plot(grp["zone_safe"], grp[metric], marker="o", label=model_name,
                color=model_colors.get(model_name), linewidth=1.5, markersize=5)
    ax.set_title(ylabel)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=35, labelsize=8)
    ax.legend(fontsize=8)
    ax.grid(axis="y", alpha=0.3)

fig.suptitle("Model performance across zones", fontsize=13, fontweight="bold")
fig.tight_layout()
chart_path = EVAL_DIR / "cross_zone_comparison.png"
fig.savefig(chart_path, dpi=180, bbox_inches="tight")
plt.show()
print(f"✓ Cross-zone chart saved → {chart_path}")
print(f"\n=== EVALUATION COMPLETE — outputs in {EVAL_DIR} ===")

=== Loading parquet ===
Raw shape: (336149, 705)
Rows ready for evaluation: 331,121
Found model bundles: ['global', 'Cold', 'Hot_Dry', 'Hot_Humid', 'Marine', 'Mixed_Dry', 'Mixed_Humid', 'Very_Cold']
[19:13:56] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0fdc6d574b9c0d168-1\xgboost\xgboost-ci-windows\src\learner.cc:553: 
  If you are loading a serialized model (like pickle in Python, RDS in R) generated by
  older XGBoost, please export the model by calling `Booster.save_model` from that version
  first, then load it back in current version. See:

    https://xgboost.readthedocs.io/en/latest/tutorials/saving_model.html

  for more details about differences between saving model and serializing.

[19:13:56] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0fdc6d574b9c0d168-1\xgboost\xgboost-ci-windows\src\learner.cc:553: 
  If you are loading a serialized model (like pickle in Python, RDS in R) generated by
  older XGBoost, plea

SHAP

In [ ]:


# ── Imports ────────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

import re
import traceback
import numpy as np
import pandas as pd
import joblib
import shap
import matplotlib
matplotlib.use("Agg")          # headless / server-safe — no Tk/Qt needed
import matplotlib.pyplot as plt
import matplotlib.cm as cm

from pathlib import Path

# ── CONFIG  — match your training script exactly ───────────────────────────────
DATA_PATH      = r"A:\College\Thesis\SHD\USstock\comstock\baseline.parquet"
MODEL_SAVE_DIR = Path(r"A:\College\Thesis\Submission\models\comstock_5.5")
SHAP_DIR       = MODEL_SAVE_DIR / "SHAP_results"
SHAP_DIR.mkdir(parents=True, exist_ok=True)

CLIMATE_COL          = "in.building_america_climate_zone"
RANDOM_STATE         = 42
USE_LOG_TARGET       = True    # must match training
APPLY_TARGET_TRIM    = True
TRIM_Q_LOW           = 0.005
TRIM_Q_HIGH          = 0.995
MIN_SAMPLES_PER_ZONE = 250

# Sampling sizes — reduce if RAM is tight
SHAP_MAX_BACKGROUND  = 500    # reference distribution for TreeExplainer
SHAP_MAX_EXPLAIN     = 1000   # rows explained (beeswarm / bar / heatmap)
SHAP_KERNEL_BG       = 100    # background rows for Stacking KernelExplainer
SHAP_KERNEL_EXPLAIN  = 200    # rows explained for Stacking (slow)
SHAP_KERNEL_NSAMPLES = 100    # KernelExplainer coalition samples per row

TOP_N_FEATURES       = 20     # features shown in beeswarm / bar
TOP_N_HEATMAP        = 15     # features shown in heatmap
TOP_N_DEPENDENCE     = 5      # dependence plots per model
TOP_N_CROSS_ZONE     = 10     # features in cross-zone grouped bar
SAVE_SHAP_VALUES     = True   # persist raw shap arrays as .joblib

DPI = 200


# ══════════════════════════════════════════════════════════════════════════════
# Helpers — mirror training exactly
# ══════════════════════════════════════════════════════════════════════════════

def safe_numeric(series):
    return pd.to_numeric(series, errors="coerce")


def parse_year_from_text(v):
    if pd.isna(v):
        return np.nan
    years = re.findall(r"(19\d{2}|20\d{2})", str(v))
    return float(max(int(y) for y in years)) if years else np.nan


def add_missing_flags(df, cols):
    for c in cols:
        if c in df.columns:
            df[f"is_missing__{c}"] = df[c].isna().astype(int)
    return df


def map_first_match(df, canonical, exact_names=None, contains_any=None):
    if canonical in df.columns:
        return canonical
    for c in (exact_names or []):
        if c in df.columns:
            df[canonical] = df[c]
            return c
    for c in df.columns:
        if all(tok in c.lower() for tok in (contains_any or [])):
            df[canonical] = df[c]
            return c
    return None


def transform_with_encoders(X_df, encoders):
    X_df = X_df.copy()
    for col in X_df.columns:
        s = X_df[col].astype("string").fillna("__MISSING__").astype(str)
        if col in encoders:
            le  = encoders[col]
            cls = set(le.classes_)
            vals = []
            for v in s.values:
                if v in cls:
                    vals.append(le.transform([v])[0])
                elif "__MISSING__" in cls:
                    vals.append(le.transform(["__MISSING__"])[0])
                else:
                    vals.append(0)
            X_df[col] = np.array(vals, dtype=np.float64)
        else:
            X_df[col] = pd.to_numeric(s, errors="coerce").fillna(0).astype(np.float64)
    return X_df


def zone_to_safe(zone_name):
    return str(zone_name).replace(" ", "_").replace("-", "_").replace("/", "_")


def shorten(name: str) -> str:
    """Strip verbose prefixes for plot readability."""
    return name.replace("out.params.", "").replace("in.", "")


def load_model_bundle(safe_zone):
    features = joblib.load(MODEL_SAVE_DIR / f"features_{safe_zone}.joblib")
    encoders = joblib.load(MODEL_SAVE_DIR / f"encoders_{safe_zone}.joblib")
    models   = {}
    for key, fname in [
        ("XGBoost",  f"xgb_{safe_zone}.joblib"),
        ("CatBoost", f"catboost_{safe_zone}.joblib"),
        ("LightGBM", f"lightgbm_{safe_zone}.joblib"),
        ("Stacking", f"stacking_{safe_zone}.joblib"),
    ]:
        p = MODEL_SAVE_DIR / fname
        if p.exists():
            models[key] = joblib.load(p)
    return features, encoders, models


# ══════════════════════════════════════════════════════════════════════════════
# Data loading & feature engineering  — mirrors training cell
# ══════════════════════════════════════════════════════════════════════════════

def build_eval_df(raw_df):
    df = raw_df.copy()

    for canon, exact, contains in [
        ("in.weekday_operating_hours",
         ["in.weekday_operating_hours", "in.weekday_operating_hours..hr"],
         ["weekday", "operating", "hour"]),
        ("in.weekend_operating_hours",
         ["in.weekend_operating_hours", "in.weekend_operating_hours..hr"],
         ["weekend", "operating", "hour"]),
        ("out.params.average_heating_setpoint_max",
         ["out.params.average_heating_setpoint_max",
          "out.params.average_heating_setpoint_max..c"],
         ["average", "heating", "setpoint", "max"]),
        ("out.params.average_heating_setpoint_min",
         ["out.params.average_heating_setpoint_min",
          "out.params.average_heating_setpoint_min..c"],
         ["average", "heating", "setpoint", "min"]),
    ]:
        map_first_match(df, canon, exact_names=exact, contains_any=contains)

    if "calc.enduse_group.site_energy.heating.energy_consumption" in df.columns:
        df["heating_energy_kwh"] = safe_numeric(
            df["calc.enduse_group.site_energy.heating.energy_consumption"])
    else:
        parts = ["out.electricity.heating.energy_consumption",
                 "out.natural_gas.heating.energy_consumption",
                 "out.other_fuel.heating.energy_consumption",
                 "out.district_heating.heating.energy_consumption"]
        df["heating_energy_kwh"] = sum(
            safe_numeric(df[c]).fillna(0.0) for c in parts if c in df.columns)

    if "in.sqft" in df.columns:
        df["floor_area_ft2"] = safe_numeric(df["in.sqft"])
    elif "calc.weighted.sqft" in df.columns:
        df["floor_area_ft2"] = safe_numeric(df["calc.weighted.sqft"])

    df = df.dropna(subset=["heating_energy_kwh", "floor_area_ft2"]).copy()
    df = df[(df["floor_area_ft2"] > 0) & (df["heating_energy_kwh"] >= 0)].copy()
    df["floor_area_m2"] = df["floor_area_ft2"] * 0.09290304
    df["specific_heat_demand_kwh_m2"] = df["heating_energy_kwh"] / df["floor_area_m2"]
    df = df[np.isfinite(df["specific_heat_demand_kwh_m2"]) &
            (df["specific_heat_demand_kwh_m2"] > 0)].copy()

    if APPLY_TARGET_TRIM:
        q1 = df["specific_heat_demand_kwh_m2"].quantile(TRIM_Q_LOW)
        q2 = df["specific_heat_demand_kwh_m2"].quantile(TRIM_Q_HIGH)
        df = df[(df["specific_heat_demand_kwh_m2"] >= q1) &
                (df["specific_heat_demand_kwh_m2"] <= q2)].copy()

    if "upgrade" in df.columns:
        u = df["upgrade"].astype("string").str.strip().str.lower()
        is_base = (u.isin(["0", "00", "baseline", "base", "upgrade00"]) |
                   u.str.fullmatch(r"0+"))
        if is_base.sum() > 1000:
            df = df[is_base].copy()

    if "in.year_built" in df.columns:
        df["year_built_num"] = df["in.year_built"].apply(parse_year_from_text)

    for c in ["in.number_of_stories", "in.aspect_ratio", "in.rotation",
              "in.weekday_operating_hours", "in.weekend_operating_hours",
              "out.params.average_heating_setpoint_max",
              "out.params.average_heating_setpoint_min"]:
        if c in df.columns:
            df[c] = safe_numeric(df[c])

    if {"in.weekday_operating_hours", "in.weekend_operating_hours"}.issubset(df.columns):
        wkd = safe_numeric(df["in.weekday_operating_hours"])
        wke = safe_numeric(df["in.weekend_operating_hours"])
        df["operating_hours_week"] = 5.0 * wkd + 2.0 * wke
        df["is_missing__in.weekday_operating_hours"] = wkd.isna().astype(int)
        df["is_missing__in.weekend_operating_hours"] = wke.isna().astype(int)

    if {"out.params.average_heating_setpoint_max",
            "out.params.average_heating_setpoint_min"}.issubset(df.columns):
        tmax = safe_numeric(df["out.params.average_heating_setpoint_max"])
        tmin = safe_numeric(df["out.params.average_heating_setpoint_min"])
        df["heating_setpoint_span_c"] = tmax - tmin
        df["is_missing__out.params.average_heating_setpoint_max"] = tmax.isna().astype(int)
        df["is_missing__out.params.average_heating_setpoint_min"] = tmin.isna().astype(int)

    if "out.params.hdd65f" in df.columns:
        hdd = safe_numeric(df["out.params.hdd65f"])
        for suffix, col in [
            ("wall_u",     "out.params.average_wall_u_value..btu_per_ft2_f_hr"),
            ("roof_u",     "out.params.average_roof_u_value"),
            ("win_u",      "out.params.average_window_u_value"),
            ("wall_area",  "out.params.ext_wall_area"),
            ("window_area","out.params.ext_window_area"),
        ]:
            if col in df.columns:
                df[f"hdd_x_{suffix}"] = hdd * safe_numeric(df[col])

    df = add_missing_flags(df, [
        "out.params.average_wall_u_value..btu_per_ft2_f_hr",
        "out.params.average_roof_u_value", "out.params.average_window_u_value",
        "out.params.hdd65f", "in.hvac_system_type", "in.window_type",
        "in.weekday_operating_hours..hr", "in.weekend_operating_hours..hr",
        "out.params.average_heating_setpoint_max..c",
        "out.params.average_heating_setpoint_min..c",
        "out.params.hours_heating_setpoint_not_met",
    ])
    return df


# ══════════════════════════════════════════════════════════════════════════════
# XGBModel compatibility patch
# ══════════════════════════════════════════════════════════════════════════════
# XGBoost >=2.x removed legacy params (gpu_id, etc.). When a StackingRegressor
# pickled with an older XGBoost is loaded and sklearn calls get_params() on
# every sub-estimator (via StackingRegressor.transform), those missing attrs
# raise AttributeError. We monkey-patch XGBModel.__getattr__ to return None for
# any attribute that is genuinely absent, restoring it immediately afterwards.

from contextlib import contextmanager

try:
    from xgboost.sklearn import XGBModel as _XGBModel
    _HAVE_XGB = True
except ImportError:
    _HAVE_XGB = False


# Exhaustive list of XGBoost sklearn-wrapper params that were removed in >=2.x
# but that older pickled models may still reference via get_params().
# We only intercept these specific names so that functional attributes like
# best_iteration are NOT silenced (silencing them causes TypeError downstream).
_XGB_REMOVED_PARAMS = frozenset({
    "gpu_id", "gpu_hist", "use_label_encoder",
    "n_jobs",                          # moved to tree_method in some versions
    "missing",                         # sometimes missing from __dict__
    "importance_type",                 # removed from some wrappers
    "interaction_constraints",
    "monotone_constraints",
    "num_parallel_tree",
    "validate_parameters",
    "enable_categorical",
    "feature_types",
    "max_cat_to_onehot",
    "max_cat_threshold",
    "grow_policy",
    "max_leaves",
    "max_bin",
    "sampling_method",
    "sketch_eps",
    "updater",
    "refresh_leaf",
    "process_type",
    "tree_method",
    "predictor",
    "single_precision_histogram",
    "deterministic_histogram",
    "callbacks",
    "early_stopping_rounds",
    "eval_metric",
    "verbosity",
})


@contextmanager
def _xgb_compat_patch():
    """
    Temporarily add a __getattr__ to XGBModel that returns None ONLY for the
    specific legacy parameter names that XGBoost >=2.x removed from its sklearn
    wrapper but that get_params() still tries to read on older pickled models.

    Functional attributes (best_iteration, best_score, etc.) are deliberately
    NOT intercepted here -- raising AttributeError for them is correct behaviour.
    """
    if not _HAVE_XGB:
        yield
        return

    original_getattr = _XGBModel.__dict__.get("__getattr__", None)

    def _selective_getattr(self, name):
        if name in _XGB_REMOVED_PARAMS:
            return None          # silently satisfy get_params() lookups
        # For everything else, raise AttributeError as normal so that XGBoost's
        # own code (e.g. best_iteration arithmetic) behaves correctly.
        raise AttributeError(
            f"'{type(self).__name__}' object has no attribute '{name}'"
        )

    _XGBModel.__getattr__ = _selective_getattr
    try:
        yield
    finally:
        if original_getattr is None:
            try:
                del _XGBModel.__getattr__
            except AttributeError:
                pass
        else:
            _XGBModel.__getattr__ = original_getattr


# ══════════════════════════════════════════════════════════════════════════════
# SHAP explainer factory
# ══════════════════════════════════════════════════════════════════════════════

def _make_predict_fn(model, use_log):
    """
    Wrap StackingRegressor.predict in a closure. Each call runs inside
    _xgb_compat_patch() so that StackingRegressor.transform -> get_params()
    does not raise AttributeError for removed XGBoost legacy params.
    """
    def _predict(X_array):
        X_np = np.asarray(X_array, dtype=np.float64)
        with _xgb_compat_patch():
            p = model.predict(X_np)
        if use_log:
            p = np.expm1(p)
            p = np.clip(p, 0, None)
        return p.astype(np.float64)
    return _predict


def _get_explainer(model, model_key, X_background, short_names, use_log):
    """
    TreeExplainer for native tree models; KernelExplainer for Stacking.
    Returns (explainer, exp_type) -- exp_type in {"tree", "kernel"}

    KernelExplainer.__init__ calls predict once (match_model_to_data), so the
    entire construction is also wrapped in _xgb_compat_patch().
    """
    if model_key in ("XGBoost", "CatBoost", "LightGBM"):
        explainer = shap.TreeExplainer(
            model,
            data=X_background,
            feature_names=short_names,
            model_output="raw",   # log-space values when USE_LOG_TARGET=True
        )
        return explainer, "tree"

    # Stacking: wrap both construction and every subsequent predict call
    predict_fn = _make_predict_fn(model, use_log)
    rng_bg     = np.random.default_rng(RANDOM_STATE)
    n_bg       = min(SHAP_KERNEL_BG, len(X_background))
    bg_subset  = X_background[rng_bg.choice(len(X_background), n_bg, replace=False)]

    with _xgb_compat_patch():
        explainer = shap.KernelExplainer(predict_fn, bg_subset)

    return explainer, "kernel"


# ══════════════════════════════════════════════════════════════════════════════
# Individual plot helpers
# ══════════════════════════════════════════════════════════════════════════════

def _save_bar(shap_values, X_explain, short_names, title, path):
    """Mean |SHAP| bar chart."""
    plt.figure(figsize=(10, max(5, TOP_N_FEATURES * 0.37)))
    shap.summary_plot(
        shap_values, features=X_explain,
        feature_names=short_names,
        max_display=TOP_N_FEATURES,
        plot_type="bar", show=False,
    )
    plt.title(title, fontsize=12, pad=10)
    plt.tight_layout()
    plt.savefig(path, dpi=DPI, bbox_inches="tight")
    plt.close("all")


def _save_beeswarm(shap_values, X_explain, short_names, title, path):
    """Dot (beeswarm) summary — shows direction and spread."""
    plt.figure(figsize=(11, max(6, TOP_N_FEATURES * 0.40)))
    shap.summary_plot(
        shap_values, features=X_explain,
        feature_names=short_names,
        max_display=TOP_N_FEATURES,
        plot_type="dot", show=False,
    )
    plt.title(title, fontsize=12, pad=10)
    plt.tight_layout()
    plt.savefig(path, dpi=DPI, bbox_inches="tight")
    plt.close("all")


def _save_heatmap(shap_values, X_explain, short_names, title, path):
    """
    Heatmap: features × samples — reveals interaction patterns.
    shap.plots.heatmap requires an Explanation object, so we build one.
    We subsample to keep the plot legible.
    """
    n_heat  = min(300, len(X_explain))
    rng_h   = np.random.default_rng(RANDOM_STATE + 1)
    idx_h   = rng_h.choice(len(X_explain), n_heat, replace=False)
    sv_heat = shap_values[idx_h]
    X_heat  = X_explain[idx_h]

    # Build Explanation object (needed by shap.plots.heatmap)
    exp = shap.Explanation(
        values=sv_heat,
        base_values=np.zeros(n_heat),   # base value not used by heatmap
        data=X_heat,
        feature_names=short_names,
    )
    shap.plots.heatmap(exp, max_display=TOP_N_HEATMAP, show=False)
    fig = plt.gcf()
    fig.set_size_inches(13, 7)
    fig.axes[0].set_title(title, fontsize=12, pad=10)
    fig.tight_layout()
    fig.savefig(path, dpi=DPI, bbox_inches="tight")
    plt.close("all")


def _save_dependence_plots(shap_values, X_explain, short_names, prefix, out_dir):
    """
    Dependence plots for top-N features by mean |SHAP|.
    Uses the legacy shap.dependence_plot (accepts ax=) so we can
    pass int indices for features that are not column names.
    """
    mean_abs     = np.abs(shap_values).mean(axis=0)
    ranked_idxs  = np.argsort(-mean_abs)
    X_df_explain = pd.DataFrame(X_explain, columns=short_names)

    plotted = 0
    for fi in ranked_idxs:
        if plotted >= TOP_N_DEPENDENCE:
            break
        fname   = short_names[fi]
        col_ref = fname if fname in X_df_explain.columns else fi
        fig, ax = plt.subplots(figsize=(8, 6))
        try:
            shap.dependence_plot(
                col_ref,
                shap_values,
                X_df_explain,
                feature_names=short_names,
                interaction_index="auto",
                ax=ax,
                show=False,
            )
        except Exception:
            plt.close(fig)
            continue
        ax.set_title(f"Dependence: {fname}", fontsize=12)
        fig.tight_layout()
        safe_fname = re.sub(r"[^\w]", "_", fname)
        fig.savefig(
            out_dir / f"{prefix}_dep_{plotted+1:02d}_{safe_fname}.png",
            dpi=DPI, bbox_inches="tight",
        )
        plt.close(fig)
        plotted += 1


def _save_waterfall(shap_values, X_explain, base_value, short_names,
                    y_series, title, path):

    preds = base_value + shap_values.sum(axis=1)
    if USE_LOG_TARGET:
        preds = np.expm1(preds)

    median_pred = np.median(preds)
    idx = int(np.argmin(np.abs(preds - median_pred)))

    # Build a single-sample Explanation
    exp_single = shap.Explanation(
        values=shap_values[idx],
        base_values=base_value,
        data=X_explain[idx],
        feature_names=short_names,
    )
    shap.plots.waterfall(exp_single, show=False)
    fig = plt.gcf()
    fig.set_size_inches(10, max(6, min(TOP_N_FEATURES, 20) * 0.38))
    actual_str = ""
    if y_series is not None and len(y_series) > idx:
        actual_str = f"  [actual = {float(np.asarray(y_series)[idx]):.1f} kWh/m²/yr]"
    fig.axes[0].set_title(f"{title}{actual_str}", fontsize=11, pad=10)
    fig.tight_layout()
    fig.savefig(path, dpi=DPI, bbox_inches="tight")
    plt.close("all")


def _save_force_html(explainer, shap_values, X_explain, short_names, path):
    """
    Interactive multi-sample force plot saved as standalone HTML.
    Capped at 200 samples for file-size reasons.
    """
    n_force = min(200, len(X_explain))
    X_df    = pd.DataFrame(X_explain[:n_force], columns=short_names)
    try:
        base = (explainer.expected_value
                if np.isscalar(explainer.expected_value)
                else float(explainer.expected_value))
        force = shap.force_plot(
            base,
            shap_values[:n_force],
            X_df,
            show=False,
        )
        shap.save_html(str(path), force)
    except Exception as exc:
        path.write_text(f"<pre>Force plot unavailable: {exc}</pre>")


# ══════════════════════════════════════════════════════════════════════════════
# Mean |SHAP| helper for cross-zone summary
# ══════════════════════════════════════════════════════════════════════════════

def _mean_abs_shap_series(shap_values, short_names):
    """Returns pd.Series: feature → mean |SHAP|, sorted descending."""
    mean_abs = np.abs(shap_values).mean(axis=0)
    return pd.Series(mean_abs, index=short_names).sort_values(ascending=False)


# ══════════════════════════════════════════════════════════════════════════════
# Per-bundle main routine
# ══════════════════════════════════════════════════════════════════════════════

def run_shap_for_bundle(safe_zone, df_zone):
    """
    Load one bundle (global or zone), compute SHAP for every model,
    save all plots, and return a list of importance dicts for the summary.
    """
    out_dir = SHAP_DIR / safe_zone
    out_dir.mkdir(parents=True, exist_ok=True)

    features, encoders, models = load_model_bundle(safe_zone)
    short_names = [shorten(f) for f in features]

    # ── Encode all rows once ───────────────────────────────────────────────
    X_raw = df_zone.reindex(columns=features).copy()
    X_raw = X_raw.astype("string").fillna("__MISSING__")
    X_df  = transform_with_encoders(X_raw, encoders)
    X_all = X_df.to_numpy(dtype=np.float64)
    y_all = df_zone["specific_heat_demand_kwh_m2"].to_numpy(dtype=np.float64)

    n   = len(X_all)
    rng = np.random.default_rng(RANDOM_STATE)

    bg_idx  = rng.choice(n, size=min(SHAP_MAX_BACKGROUND, n), replace=False)
    X_background = X_all[bg_idx]

    importance_rows = []

    for model_key, model in models.items():
        print(f"  [{safe_zone}] SHAP → {model_key} …", flush=True)
        prefix = f"{model_key.lower()}"

        try:
            # ── Explainer ─────────────────────────────────────────────────
            explainer, exp_type = _get_explainer(
                model, model_key, X_background, short_names, USE_LOG_TARGET)

            # ── Sample rows to explain ─────────────────────────────────────
            max_exp = SHAP_KERNEL_EXPLAIN if exp_type == "kernel" else SHAP_MAX_EXPLAIN
            exp_idx = rng.choice(n, size=min(max_exp, n), replace=False)
            X_explain = X_all[exp_idx]
            y_explain = y_all[exp_idx]

            # ── Compute SHAP values ────────────────────────────────────────
            if exp_type == "tree":
                shap_values = explainer.shap_values(X_explain, check_additivity=False)
            else:
                shap_values = explainer.shap_values(
                    X_explain, nsamples=SHAP_KERNEL_NSAMPLES, silent=True)

            if isinstance(shap_values, list):
                shap_values = shap_values[0]

            # Scalar base value for plots
            base_val = (explainer.expected_value
                        if np.isscalar(explainer.expected_value)
                        else float(np.squeeze(explainer.expected_value)))

            # ── 1. Bar ─────────────────────────────────────────────────────
            _save_bar(
                shap_values, X_explain, short_names,
                f"SHAP Feature Importance — {safe_zone} — {model_key}",
                out_dir / f"{prefix}_bar.png",
            )

            # ── 2. Beeswarm ────────────────────────────────────────────────
            _save_beeswarm(
                shap_values, X_explain, short_names,
                f"SHAP Beeswarm — {safe_zone} — {model_key}",
                out_dir / f"{prefix}_beeswarm.png",
            )

            # ── 3. Heatmap ─────────────────────────────────────────────────
            _save_heatmap(
                shap_values, X_explain, short_names,
                f"SHAP Heatmap — {safe_zone} — {model_key}",
                out_dir / f"{prefix}_heatmap.png",
            )

            # ── 4. Dependence plots (top-5) ────────────────────────────────
            _save_dependence_plots(
                shap_values, X_explain, short_names,
                prefix, out_dir,
            )

            # ── 5. Waterfall (median-prediction sample) ────────────────────
            _save_waterfall(
                shap_values, X_explain, base_val, short_names,
                y_explain,
                f"SHAP Local Explanation (median pred) — {safe_zone} — {model_key}",
                out_dir / f"{prefix}_waterfall.png",
            )

            # ── 6. Force plot (interactive HTML) ──────────────────────────
            _save_force_html(
                explainer, shap_values, X_explain, short_names,
                out_dir / f"{prefix}_force.html",
            )

            # ── Optionally persist raw arrays ──────────────────────────────
            if SAVE_SHAP_VALUES:
                joblib.dump(
                    {"shap_values": shap_values, "X_explain": X_explain,
                     "base_value": base_val, "feature_names": short_names},
                    out_dir / f"{prefix}_shap_values.joblib",
                )

            # ── Collect for cross-zone summary ─────────────────────────────
            imp = _mean_abs_shap_series(shap_values, short_names)
            row = {"model": model_key, "scope": safe_zone}
            row.update(imp.to_dict())
            importance_rows.append(row)

            print(f"    ✓ bar | beeswarm | heatmap | dep×{TOP_N_DEPENDENCE}"
                  f" | waterfall | force.html")

        except Exception:
            print(f"    ✗ ERROR for {model_key} in {safe_zone}:")
            traceback.print_exc()

    return importance_rows


# ══════════════════════════════════════════════════════════════════════════════
# Cross-zone summary plots
# ══════════════════════════════════════════════════════════════════════════════

def build_cross_zone_summary(all_importance_rows):
    if not all_importance_rows:
        print("No importance data collected — skipping cross-zone summary.")
        return

    importance_df = (
        pd.DataFrame(all_importance_rows)
          .set_index(["model", "scope"])
          .fillna(0.0)
    )

    # ── 7. CSV export ──────────────────────────────────────────────────────
    csv_path = SHAP_DIR / "shap_importance_all_models.csv"
    importance_df.to_csv(csv_path)
    print(f"\n  CSV saved → {csv_path.name}")

    # ── 8. Grouped bar chart: top-N features across zones ─────────────────
    scope_importance = importance_df.groupby("scope").mean()
    global_mean      = scope_importance.mean(axis=0).sort_values(ascending=False)
    top_features     = global_mean.head(TOP_N_CROSS_ZONE).index.tolist()
    scope_top        = scope_importance[top_features]

    n_scopes   = len(scope_top)
    n_features = len(top_features)
    bar_width  = max(0.06, 0.7 / max(n_scopes, 1))
    colors     = cm.tab20(np.linspace(0, 1, max(n_scopes, 1)))

    fig, ax = plt.subplots(figsize=(14, 7))
    x = np.arange(n_features)
    for i, (scope_name, row_vals) in enumerate(scope_top.iterrows()):
        offset = (i - n_scopes / 2 + 0.5) * bar_width
        ax.bar(
            x + offset, row_vals[top_features].values,
            bar_width, label=scope_name,
            color=colors[i], alpha=0.88,
        )

    ax.set_xticks(x)
    ax.set_xticklabels(
        [shorten(f) for f in top_features],
        rotation=35, ha="right", fontsize=9,
    )
    ax.set_ylabel("Mean |SHAP value| (kWh/m²/yr)")
    ax.set_title(
        f"Top-{TOP_N_CROSS_ZONE} Feature Importance by Scope"
        " (mean across model types)",
        fontsize=12,
    )
    ax.legend(loc="upper right", fontsize=8, ncol=2)
    ax.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    cross_path = SHAP_DIR / "shap_cross_zone_comparison.png"
    fig.savefig(cross_path, dpi=DPI, bbox_inches="tight")
    plt.close(fig)
    print(f"  Cross-zone chart saved → {cross_path.name}")

    # ── Text summary ───────────────────────────────────────────────────────
    global_rank = importance_df.mean(axis=0).sort_values(ascending=False).head(10)
    max_v = global_rank.iloc[0] if len(global_rank) else 1.0
    print("\nTOP-10 FEATURES  (averaged across all models & zones)")
    print("─" * 62)
    for rank, (feat, val) in enumerate(global_rank.items(), 1):
        bar = "█" * int(val / max_v * 30)
        print(f"  {rank:2d}. {shorten(feat):<45s} {val:6.3f}  {bar}")


# ══════════════════════════════════════════════════════════════════════════════
# Main
# ══════════════════════════════════════════════════════════════════════════════

print("=== Loading parquet ===")
raw_df = pd.read_parquet(DATA_PATH).convert_dtypes()
print(f"Raw shape: {raw_df.shape}")

df_all = build_eval_df(raw_df)
df_all = df_all.dropna(subset=["specific_heat_demand_kwh_m2", CLIMATE_COL]).copy()
print(f"Rows after preprocessing: {len(df_all)}")

# Discover all bundles by scanning saved feature files
bundle_safe_names = [
    p.stem.replace("features_", "", 1)
    for p in sorted(MODEL_SAVE_DIR.glob("features_*.joblib"))
]
print(f"\nBundles found: {bundle_safe_names}\n")

# Reverse map: safe_zone string → raw zone label in parquet
zone_labels      = sorted(df_all[CLIMATE_COL].astype("string").dropna().unique())
safe_to_raw_zone = {zone_to_safe(z): z for z in zone_labels}

all_importance_rows = []

for safe_zone in bundle_safe_names:
    print(f"\n{'='*62}")
    print(f"  Bundle: {safe_zone}")
    print(f"{'='*62}")

    if safe_zone == "global":
        df_zone = df_all
    else:
        raw_zone = safe_to_raw_zone.get(safe_zone)
        if raw_zone is None:
            print(f"  ⚠  No data rows for '{safe_zone}', skipping.")
            continue
        df_zone = df_all[df_all[CLIMATE_COL].astype("string") == raw_zone].copy()
        if len(df_zone) < MIN_SAMPLES_PER_ZONE:
            print(f"  ⚠  Only {len(df_zone)} rows < {MIN_SAMPLES_PER_ZONE}, skipping.")
            continue

    rows = run_shap_for_bundle(safe_zone, df_zone)
    all_importance_rows.extend(rows)

# Cross-zone summary
print(f"\n{'='*62}")
print("  Cross-zone summary")
print(f"{'='*62}")
build_cross_zone_summary(all_importance_rows)

print(f"\n=== SHAP ANALYSIS COMPLETE ===")
print(f"All outputs saved under: {SHAP_DIR.resolve()}")

=== Loading parquet ===
Raw shape: (336149, 705)
Rows ready for evaluation: 331,121
Found model bundles: ['global', 'Cold', 'Hot_Dry', 'Hot_Humid', 'Marine', 'Mixed_Dry', 'Mixed_Humid', 'Very_Cold']
[16:27:01] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0fdc6d574b9c0d168-1\xgboost\xgboost-ci-windows\src\learner.cc:553: 
  If you are loading a serialized model (like pickle in Python, RDS in R) generated by
  older XGBoost, please export the model by calling `Booster.save_model` from that version
  first, then load it back in current version. See:

    https://xgboost.readthedocs.io/en/latest/tutorials/saving_model.html

  for more details about differences between saving model and serializing.

    [xgb] Extracted booster via save_model (top-level)
[16:27:01] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0fdc6d574b9c0d168-1\xgboost\xgboost-ci-windows\src\learner.cc:553: 
  If you are loading a serialized model (like pickle 